# Upload Dataset & Models to Hugging Face Hub

Uploads the parallel corpus, dialect dictionaries, and fine-tuned models for the Лексикон project.

In [1]:
HF_USERNAME = "DenysKovalML"          # change this
DATASET_REPO = f"{HF_USERNAME}/ukrainian-dialect-normalization"
DICTS_REPO   = f"{HF_USERNAME}/ukrainian-dialect-dictionaries"
UMT5_REPO    = f"{HF_USERNAME}/umt5-base-ukrainian-dialect-normalization"
MAMAY_REPO   = f"{HF_USERNAME}/mamaylm-ukrainian-dialect-normalization"

PRIVATE = False   # set True to keep repos private

In [ ]:
from datasets import DatasetDict, Dataset
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import re
from huggingface_hub import HfApi

## 1. Parallel corpus

In [ ]:
DATA_DIR = "../../data/parallel"

DIALECT_PREFIXES = {
    "гуцульської": "hutsul",
    "бойківської": "boiko",
    "закарпатської": "transcarpathian",
    "суржику": "surzhyk",
}

def add_dialect_column(df: pd.DataFrame) -> pd.DataFrame:
    def extract(text):
        m = re.search(r"Переклади з (\w+):", text)
        if m:
            return DIALECT_PREFIXES.get(m.group(1), m.group(1))
        return None
    df = df.copy()
    df["dialect"] = df["source"].apply(extract)
    return df

train_df = add_dialect_column(pd.read_csv(f"{DATA_DIR}/train.csv"))
val_df   = add_dialect_column(pd.read_csv(f"{DATA_DIR}/val.csv"))
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

dataset = DatasetDict({
    "train":      Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df,   preserve_index=False),
    "test":       Dataset.from_pandas(test_df,  preserve_index=False),
})

print(dataset)
dataset.push_to_hub(DATASET_REPO, private=PRIVATE)

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'dialect'],
        num_rows: 125608
    })
    validation: Dataset({
        features: ['source', 'target', 'dialect'],
        num_rows: 13755
    })
    test: Dataset({
        features: ['source', 'target', 'dialect'],
        num_rows: 200
    })
})


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/126 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/DenysKovalML/ukrainian-dialect-normalization/commit/70c92505f49fa409e7fd9d72df188ab363e20de3', commit_message='Upload dataset', commit_description='', oid='70c92505f49fa409e7fd9d72df188ab363e20de3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DenysKovalML/ukrainian-dialect-normalization', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DenysKovalML/ukrainian-dialect-normalization'), pr_revision=None, pr_num=None)

## 2. Dialect dictionaries

In [6]:
DICT_FILES = {
    "hutsul":         "../../data/dicts/hutsul_ukrainian_dictionary.csv",
    "boykivian":      "../../data/dicts/boykivian_ukrainian_dictionary.csv",
    "transcarpathian":"../../data/dicts/transcarpathian_ukrainian_dictionary.csv",
    "surzhyk":        "../../data/dicts/surzhyk_ukrainian_dictionary.csv",
}

def load_dict(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    dialect_col = [c for c in df.columns if c not in ("Ukrainian", "uk_lemma")][0]
    df = df.rename(columns={dialect_col: "dialect_form"})
    if "uk_lemma" not in df.columns:
        df["uk_lemma"] = pd.Series(dtype="string")
    return df[["dialect_form", "Ukrainian", "uk_lemma"]]

dicts_dataset = DatasetDict({
    name: Dataset.from_pandas(load_dict(path), preserve_index=False)
    for name, path in DICT_FILES.items()
})

print(dicts_dataset)
dicts_dataset.push_to_hub(DICTS_REPO, private=PRIVATE)

DatasetDict({
    hutsul: Dataset({
        features: ['dialect_form', 'Ukrainian', 'uk_lemma'],
        num_rows: 7319
    })
    boykivian: Dataset({
        features: ['dialect_form', 'Ukrainian', 'uk_lemma'],
        num_rows: 14909
    })
    transcarpathian: Dataset({
        features: ['dialect_form', 'Ukrainian', 'uk_lemma'],
        num_rows: 7468
    })
    surzhyk: Dataset({
        features: ['dialect_form', 'Ukrainian', 'uk_lemma'],
        num_rows: 569
    })
})


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/DenysKovalML/ukrainian-dialect-dictionaries/commit/033b4ed19037eb2ce8d8068f478ee1f1f42ae294', commit_message='Upload dataset', commit_description='', oid='033b4ed19037eb2ce8d8068f478ee1f1f42ae294', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DenysKovalML/ukrainian-dialect-dictionaries', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DenysKovalML/ukrainian-dialect-dictionaries'), pr_revision=None, pr_num=None)

## 3. umt5-base (encoder-decoder, full fine-tune)

In [ ]:
MODEL_PATH = "../../models/umt5-base-multidialect-longer/final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

tokenizer.push_to_hub(UMT5_REPO, private=PRIVATE)
model.push_to_hub(UMT5_REPO, private=PRIVATE)

## 4. MamayLM (Gemma-3-4B + QLoRA adapter)

In [ ]:
ADAPTER_PATH = "../../models/mamaylm-multidialect-longer"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.push_to_hub(MAMAY_REPO, private=PRIVATE)

api = HfApi()
api.create_repo(MAMAY_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=MAMAY_REPO,
    repo_type="model",
    ignore_patterns=["checkpoint-*", "runs/", "*.bin"],
)